# Лабораторная работа №1
## Основы парадигмы MapReduce: Анализ логов веб-сервера
**Цель работы:** Освоить принципы распределённой обработки данных на примере парадигмы MapReduce.


In [ ]:
import random
import os

def generate_logs(filename, num_lines):
    ips = [f"192.168.1.{i}" for i in range(1, 20)] + [f"10.0.0.{i}" for i in range(1, 10)]
    with open(filename, 'w') as f:
        for i in range(num_lines):
            ip = random.choice(ips)
            if random.random() < 0.1:
                ip = "1.1.1.1"
            f.write(f'{ip} - - [15/Oct/2023:14:23:45 +0300] "GET /api/v1/users HTTP/1.1" 200 3425 "-" "-"\n')

generate_logs('access_small.log', 1000)
generate_logs('access.log', 100000) # Используем 100k строк для быстрого тестирования в Colab
print("Тестовые данные сгенерированы.")

In [ ]:
import time
from collections import Counter
import tracemalloc

def extract_ip(line):
    return line.split()[0] if line.strip() else None

def naive_solution(filename):
    start_time = time.time()
    ip_counter = Counter()
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            ip = extract_ip(line)
            if ip:
                ip_counter[ip] += 1
    top_10 = ip_counter.most_common(10)
    return top_10, time.time() - start_time


In [ ]:
def mapper(chunk):
    local_count = {}
    for line in chunk:
        ip = extract_ip(line)
        if ip:
            local_count[ip] = local_count.get(ip, 0) + 1
    return local_count.items()

def mapreduce_solution(filename, chunk_size=10000):
    start_time = time.time()

    def read_chunks():
        with open(filename, 'r', encoding='utf-8') as f:
            chunk = []
            for line in f:
                chunk.append(line)
                if len(chunk) >= chunk_size:
                    yield chunk
                    chunk = []
            if chunk:
                yield chunk

    mapped_results = []
    for chunk in read_chunks():
        mapped_results.extend(mapper(chunk))

    shuffled = {}
    for ip, count in mapped_results:
        if ip not in shuffled:
            shuffled[ip] = []
        shuffled[ip].append(count)

    reduced = {}
    for ip, counts in shuffled.items():
        reduced[ip] = sum(counts)

    top_10 = sorted(reduced.items(), key=lambda x: x[1], reverse=True)[:10]
    return top_10, time.time() - start_time

In [ ]:
def measure_memory_usage(solution_func, filename):
    tracemalloc.start()
    result, execution_time = solution_func(filename)
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, execution_time, peak / 1024 / 1024

files_to_test = [
    ('access_small.log', 'Маленький файл (1K строк)'),
    ('access.log', 'Большой файл (100K строк)')
]

for filename, description in files_to_test:
    print(f"\n{'='*60}\nТестирование: {description}\n{'='*60}")
    if os.path.exists(filename):
        top_naive, time_naive, mem_naive = measure_memory_usage(naive_solution, filename)
        print(f"\nНаивное решение (время: {time_naive:.4f} сек, память: {mem_naive:.2f} MB):")
        for i, (ip, count) in enumerate(top_naive, 1):
            print(f"  {i:2}. {ip:15} - {count:6} запросов")

        top_mr, time_mr, mem_mr = measure_memory_usage(mapreduce_solution, filename)
        print(f"\nMapReduce решение (время: {time_mr:.4f} сек, память: {mem_mr:.2f} MB):")
        for i, (ip, count) in enumerate(top_mr, 1):
            print(f"  {i:2}. {ip:15} - {count:6} запросов")

        print(f"\nСравнение производительности:")
        if time_naive > 0:
             print(f"  MapReduce отношение времени: {time_mr/time_naive:.2f}x")


### Ответы на контрольные вопросы:
1. **В чём основное отличие парадигмы MapReduce от традиционной обработки данных?**
Традиционная (наивная) обработка данных обычно работает в одном потоке и загружает данные последовательно, что может привести к нехватке оперативной памяти при очень больших файлах. Парадигма MapReduce позволяет разбить обработку на небольшие независимые блоки (этап Map), которые можно вычислять параллельно, после чего результаты группируются и агрегируются (этап Reduce).

2. **Какие преимущества даёт этап Shuffle в алгоритме MapReduce?**
Этап Shuffle группирует все пары "ключ-значение" по ключам (в нашем случае — по IP-адресам). Это гарантирует, что функция Reduce получит сразу все значения, связанные с одним ключом, независимо от того, какие именно узлы или процессы (Mapper'ы) их сгенерировали.

3. **Почему "наивное" решение может не работать на очень больших файлах?**
В зависимости от реализации, наивное решение может попытаться загрузить весь файл целиком в оперативную память. Даже если оно читает файл построчно, итоговый словарь со всеми уникальными IP-адресами (или другими ключами) может разрастись и исчерпать доступную ОЗУ.

4. **Как можно распараллелить выполнение функций Map в данной лабораторной работе?**
Для этого можно использовать стандартные библиотеки Python, такие как `multiprocessing` (например, `Pool.map`) или `concurrent.futures`. Файл разбивается на чанки, и каждый чанк отправляется в отдельный процесс, где над ним выполняется функция `mapper`.

5. **Какие изменения нужно внести в программу для анализа других параметров лога (например, статус-кодов ответов)?**
Потребуется изменить логику парсинга в функции `extract_ip`. Поскольку статус-код находится в другой части строки лога, можно использовать регулярные выражения (модуль `re`) для поиска числа, идущего после HTTP-запроса, и извлекать его в качестве ключа для Map-фазы.
